In [1]:
import ecco_access as ea
import xarray as xr
import matplotlib.pyplot as plt
import os
from os.path import join,expanduser
import numpy as np
# import ecco_v4_py as ecco
import ecco_access as ea

In [2]:
#https://raw.githubusercontent.com/ECCO-GROUP/ECCO-ACCESS/refs/heads/main/varlist/v4r4/v4r4_nctiles_monthly_varlist.txt
#https://raw.githubusercontent.com/ECCO-GROUP/ECCO-ACCESS/refs/heads/main/varlist/v4r4/v4r4_nctiles_snapshots_varlist.txt

In [12]:
datadir = os.path.join('..', 'data')
datadir = os.path.abspath(os.path.normpath(datadir))
savedir = lambda x: os.path.join(datadir, x)

def get_datasets(dslist):
    return ea.ecco_podaac_to_xrdataset(dslist,\
                                        StartDate='1994-01',EndDate='1994-02',\
                                        mode='download',\
                                        grid = "native",
                                        time_res = "all",
                                        download_root_dir=savedir('ECCO_V4r4_PODAAC'))
def llc90_plotter(fig, ax, ds): 
    vmin = ds.min()
    vmin = ds.max()
    for t in ds.tile: 
        dst = ds.sel(tile = t)
        ax.pcolormesh(dst.XC, dst.YC, dst, vmin = vmin, vmax = vmax)


In [17]:
# download data and open xarray dataset
# geometry_shortname = 'ECCO_L4_GEOMETRY_LLC0090GRID_V4R4' this download doesnt work atm 

ds_grid = xr.open_dataset(savedir("ECCO_V4r4_PODAAC/GRID_GEOMETRY_ECCO_V4r4_native_llc0090.nc"))
ds_grid = ds_grid.assign_coords(ds_grid.data_vars)

# ssh_snapshot_shortname = 'ECCO_L4_SSH_LLC0090GRID_SNAPSHOT_V4R4'
# ts_snap_shortname = 'ECCO_L4_TEMP_SALINITY_LLC0090GRID_SNAPSHOT_V4R4'
# ds_budget_snap = get_datasets([ssh_snapshot_shortname,ts_snap_shortname])


shortname_list= ['ECCO_L4_OCEAN_3D_VOLUME_FLUX_LLC0090GRID_MONTHLY_V4R4', 
                'ECCO_L4_OCEAN_3D_TEMPERATURE_FLUX_LLC0090GRID_MONTHLY_V4R4', 
                'ECCO_L4_FRESH_FLUX_LLC0090GRID_MONTHLY_V4R4']
ds_monthly_dict = get_datasets(shortname_list)

shortname_list= ['ECCO_L4_FRESH_FLUX_LLC0090GRID_MONTHLY_V4R4', 
                'ECCO_L4_HEAT_FLUX_LLC0090GRID_MONTHLY_V4R4',]
ds_flux_dict = get_datasets(shortname_list)

Download to directory /Users/anthonymeza/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Documents/GitHub/xbudget/data/ECCO_V4r4_PODAAC/ECCO_L4_OCEAN_3D_VOLUME_FLUX_LLC0090GRID_MONTHLY_V4R4

OCEAN_3D_VOLUME_FLUX_mon_mean_1994-02_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading
OCEAN_3D_VOLUME_FLUX_mon_mean_1994-01_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading

DL Progress: 100%|#########################| 2/2 [00:00<00:00, 5573.83it/s]

total downloaded: 0.0 Mb
avg download speed: 0.0 Mb/s
Time spent = 0.004923343658447266 seconds


Download to directory /Users/anthonymeza/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Documents/GitHub/xbudget/data/ECCO_V4r4_PODAAC/ECCO_L4_OCEAN_3D_TEMPERATURE_FLUX_LLC0090GRID_MONTHLY_V4R4

OCEAN_3D_TEMPERATURE_FLUX_mon_mean_1994-01_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading

OCEAN_3D_TEMPERATURE_FLUX_mon_mean_1994-02_

In [ ]:
ssh_snapshot_shortname = 'ECCO_L4_SSH_LLC0090GRID_SNAPSHOT_V4R4'
ts_snap_shortname = 'ECCO_L4_TEMP_SALINITY_LLC0090GRID_SNAPSHOT_V4R4'
get_datasets([ssh_snapshot_shortname,ts_snap_shortname])

snap_terms = ["ETAN", "THETA"]
ds_budget_snap = xr.merge([v for v in ds_snapshot_dict.values()], compat='no_conflicts')[snap_terms]
ds_budget_snap = ds_budget_snap.rename({var:f"{var}_bounds" for var in ds_budget_snap.data_vars})
ds_budget_snap = ds_budget_snap.rename({"time":"time_bounds"})

monthly_terms = ["UVELMASS", "VVELMASS", "WVELMASS", "ADVx_TH", 
                "ADVy_TH", "ADVr_TH", "DFxE_TH", "DFyE_TH", 
                "DFrE_TH", "DFrI_TH"]
ds_budget_monthly = xr.merge([v for v in ds_monthly_dict.values()], compat='no_conflicts')[monthly_terms]

flux_terms = ["oceFWflx", "TFLUX", "oceQsw"]
ds_flux_monthly = xr.merge([v for v in ds_flux_dict.values()], compat='no_conflicts')[flux_terms]


In [ ]:
ds_budget = xr.merge([ds_grid, ds_budget_snap, ds_budget_monthly, ds_flux_monthly], compat='no_conflicts')


In [ ]:
ds_budget

In [ ]:
import ecco_v4_py as ecco

def calc_3d_geothermal_flux(ds):
    def read_geothermal_fluxes(ds): 
        import os
        import tempfile
        import requests
        # geoflx = ecco.read_llc_to_tiles("/efs_ecco/ECCO/V4/r4/input/input_forcing/other", 'geothermalFlux.bin')
        url = "https://github.com/ECCO-GROUP/ECCO-v4-Python-Tutorial/raw/master/misc/geothermalFlux.bin"

        with tempfile.TemporaryDirectory() as td:
            fp = os.path.join(td, "geothermalFlux.bin")

            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(fp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            geoflx = ecco.read_llc_to_tiles(td, "geothermalFlux.bin")

            # Convert numpy array to an xarray DataArray with matching dimensions as the monthly mean fields
            geoflx_llc = xr.DataArray(geoflx,coords={'tile': ds.tile.values,
                                                    'j': ds.j.values,
                                                    'i': ds.i.values},dims=['tile','j','i'])
        return geoflx_llc

    volcello = (ds["drF"] * ds["hFacC"]) * ds["rA"]
    # Seawater density (kg/m^3)
    rhoconst = 1029.0
    ## needed to convert surface mass fluxes to volume fluxes
    
    # Heat capacity (J/kg/K)
    c_p = 3994.0
    
    # Create 3d bathymetry mask
    mskC = 1 * ds["hFacC"].copy(deep=True).compute()
    
    mskC_shifted = mskC.shift(k=-1)
    
    mskC_shifted.values[-1,:,:,:] = 0
    mskb = mskC - mskC_shifted

    geoflx_llc = read_geothermal_fluxes(ds)
    # Create 3d field of geothermal heat flux
    geoflx3d = geoflx_llc * mskb.transpose('k','tile','j','i')
    GEOFLX = geoflx3d.transpose('k','tile','j','i')
    GEOFLX.attrs = {'standard_name': 'GEOFLX','long_name': 'Geothermal heat flux','units': 'W/m^2'}
    
    # Add geothermal heat flux to forcing field and convert from W/m^2 to degC/s
    G_geothermal_forcing = ((GEOFLX)/(rhoconst*c_p))/(ds["hFacC"]*ds.drF)

    volume_weighted_geothermal = G_geothermal_forcing * volcello
    volume_weighted_geothermal.attrs = {'standard_name': 'geothermal_heat_flux_convergence',
                                        'long_name': 'Geothermal heat flux convergence',
                                        'units': 'degree_C m3 s-1'}
        
    return volume_weighted_geothermal.fillna(0.0)

def calc_3d_surf_heat_flux(ds):
    volcello = (ds["drF"] * ds["hFacC"]) * ds["rA"]

    # Seawater density (kg/m^3)
    rhoconst = 1029
    ## needed to convert surface mass fluxes to volume fluxes
    
    # Heat capacity (J/kg/K)
    c_p = 3994
    
    # Constants for surface heat penetration (from Table 2 of Paulson and Simpson, 1977)
    R = 0.62
    zeta1 = 0.6
    zeta2 = 20.0
    
    Z = ds["Z"].compute()
    RF = np.concatenate([ds.Zp1.values[:-1],[np.nan]])
    q1 = R*np.exp(1.0/zeta1*RF[:-1]) + (1.0-R)*np.exp(1.0/zeta2*RF[:-1])
    q2 = R*np.exp(1.0/zeta1*RF[1:]) + (1.0-R)*np.exp(1.0/zeta2*RF[1:])
    # Create xarray data arrays
    q1 = xr.DataArray(q1,coords=[Z.k],dims=['k'])
    q2 = xr.DataArray(q2,coords=[Z.k],dims=['k'])
    
    # Correction for the 200m cutoff
    zCut = np.where(Z < -200)[0][0]
    q1[zCut:] = 0
    q2[zCut-1:] = 0
    
    ## Land masks
    # Make copy of hFacC
    mskC = 1 * ds["hFacC"].copy(deep=True).compute()
    
    # Change all fractions (ocean) to 1. land = 0
    mskC.values[mskC.values>0] = 1
    
    # Shortwave flux below the surface (W/m^2)
    forcH_subsurf = ((q1*(mskC==1)-q2*(mskC.shift(k=-1)==1))*ds["oceQsw"]).transpose('time','tile','k','j','i')
    
    # Surface heat flux (W/m^2) + shortwave convergence at surface
    forcH_surf = ((ds["TFLUX"] - (1-(q1[0]-q2[0]))*ds["oceQsw"])\
                  *mskC[0]).transpose('time','tile','j','i').assign_coords(k=0).expand_dims('k')
    
    # Full-depth sea surface forcing (W/m^2)
    forcH = xr.concat([forcH_surf,forcH_subsurf[:,:,1:]], dim='k').transpose('time','tile','k','j','i')

    # Add geothermal heat flux to forcing field and convert from W/m^2 to degC/s
    G_surf_forcing = ((forcH)/(rhoconst*c_p))/(ds.hFacC*ds.drF)
    

    volume_weighted_surf_forcing = G_surf_forcing * volcello
    volume_weighted_surf_forcing.attrs = {'standard_name': 'boundary_forcing_heat_tendency',
                                        'long_name': 'boundary_forcing_heat_tendency',
                                        'units': 'degree_C m3 s-1'}
        
    return volume_weighted_surf_forcing.fillna(0.0)



In [ ]:
ds_budget["boundary_forcing_heat_tendency"] = calc_3d_surf_heat_flux(ds_budget).fillna(0.0)
ds_budget["geothermal_heat_flux_convergence"] = calc_3d_geothermal_flux(ds_budget).fillna(0.0)

In [ ]:
ds_budget.chunk("auto").to_zarr("../data/data/ECCO_budget_terms.zarr", compute = True, mode = "w")